In [ ]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
import time
from datetime import datetime, timezone

# Project paths
PROJECT_ROOT = Path.home() / "wspr-propagation"
WSPR_DATA_DIR = PROJECT_ROOT / "data" / "wspr"
WSPR_DATA_DIR.mkdir(parents=True, exist_ok=True)

# WSPR live endpoint
WSPR_URL = "https://db1.wspr.live/"

# Bands of interest (in meters)
BANDS = [10, 20, 40]

# Geographic bounding box — North America + Europe
# tx or rx must fall within this box
LAT_MIN, LAT_MAX = 25.0, 70.0
LON_MIN, LON_MAX = -130.0, 40.0

print(f"Data directory: {WSPR_DATA_DIR}")
print(f"Bands: {BANDS}m")

In [ ]:
def fetch_wspr_day(date: str, band: int) -> pd.DataFrame:
    """
    Fetch one day of WSPR spots for a given band.
    date: 'YYYY-MM-DD'
    band: band code (28=10m, 14=20m, 7=40m)
    """
    
    query = f"""
        SELECT
            time,
            tx_sign,
            rx_sign,
            tx_loc,
            rx_loc,
            tx_lat,
            tx_lon,
            rx_lat,
            rx_lon,
            distance,
            band,
            frequency,
            power,
            snr,
            drift
        FROM wspr.rx
        WHERE
            date(time) = '{date}'
            AND band = {band}
            AND tx_lat BETWEEN {LAT_MIN} AND {LAT_MAX}
            AND tx_lon BETWEEN {LON_MIN} AND {LON_MAX}
            AND rx_lat BETWEEN {LAT_MIN} AND {LAT_MAX}
            AND rx_lon BETWEEN {LON_MIN} AND {LON_MAX}
        FORMAT JSONCompact
    """
    
    try:
        response = requests.get(
            WSPR_URL,
            params={"query": query},
            timeout=60
        )
        response.raise_for_status()
        
        data = response.json()
        
        if not data.get("data"):
            print(f"  No data returned for {date} band={band}")
            return pd.DataFrame()
        
        cols = [col["name"] for col in data["meta"]]
        df = pd.DataFrame(data["data"], columns=cols)
        
        df["time"] = pd.to_datetime(df["time"])
        for col in ["tx_lat","tx_lon","rx_lat","rx_lon","frequency","snr","drift"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df["power"] = pd.to_numeric(df["power"], errors="coerce")
        df["distance"] = pd.to_numeric(df["distance"], errors="coerce")
        df["band"] = pd.to_numeric(df["band"], errors="coerce")
        
        return df
    
    except Exception as e:
        print(f"  Error fetching {date} band={band}: {e}")
        return pd.DataFrame()

# Band code mapping
BAND_CODES = {10: 28, 20: 14, 40: 7}

print("fetch_wspr_day() redefined")
print(f"Band codes: {BAND_CODES}")

In [ ]:
test_date = "2023-06-15"
test_band = BAND_CODES[20]

print(f"Fetching {test_date} band code {test_band} (20m) ...")
df_test = fetch_wspr_day(test_date, test_band)

if not df_test.empty:
    print(f"\nRows: {len(df_test):,}")
    print(f"Columns: {list(df_test.columns)}")
    print(f"Time range: {df_test['time'].min()} → {df_test['time'].max()}")
    print(f"SNR range: {df_test['snr'].min():.0f} to {df_test['snr'].max():.0f} dB")
    print(f"Power range: {df_test['power'].min():.0f} to {df_test['power'].max():.0f} dBm")
    print(f"Distance range: {df_test['distance'].min():.0f} to {df_test['distance'].max():.0f} km")
    display(df_test.head())

In [ ]:
def day_parquet_path(date: str, band_m: int) -> Path:
    """Return the parquet path for a given date and band in meters."""
    return WSPR_DATA_DIR / f"wspr_{date}_{band_m}m.parquet"

def fetch_and_cache_day(date: str, band_m: int, force: bool = False) -> pd.DataFrame:
    """
    Fetch a day of WSPR data and cache to parquet.
    Skips fetch if parquet already exists unless force=True.
    band_m: band in meters (10, 20, 40)
    """
    path = day_parquet_path(date, band_m)
    
    if path.exists() and not force:
        return pd.read_parquet(path)
    
    band_code = BAND_CODES.get(band_m)
    if band_code is None:
        raise ValueError(f"Unknown band: {band_m}m. Add to BAND_CODES dict.")
    
    print(f"  Fetching {date} {band_m}m ...", end=" ")
    df = fetch_wspr_day(date, band_code)
    
    if df.empty:
        print("empty.")
        return df
    
    # Basic quality filters
    df = df[df["distance"] > 0]           # remove self-spots
    df = df[df["snr"].between(-35, 20)]   # remove implausible SNR values
    df = df[df["power"].between(0, 57)]   # 1mW to 500W
    df = df[df["drift"].abs() <= 4]       # remove unstable transmitters
    
    # Compute path loss: SNR is relative to noise floor, 
    # but tx_power - snr gives a relative path loss metric
    # Full link budget requires noise floor assumption — we'll add that later
    df["path_loss_proxy"] = df["power"] - df["snr"]
    
    df.to_parquet(path, index=False)
    print(f"{len(df):,} rows cached.")
    return df

# Test the caching layer with our already-fetched test date
print("Testing cache layer...")
df_cached = fetch_and_cache_day("2023-06-15", 20)
print(f"Loaded {len(df_cached):,} rows")
print(f"Path loss proxy range: {df_cached['path_loss_proxy'].min():.0f} to {df_cached['path_loss_proxy'].max():.0f} dB")

In [ ]:
from datetime import date, timedelta

def third_monday(year: int, month: int) -> date:
    """Return the third Monday of a given month."""
    d = date(year, month, 1)
    # Find first Monday
    days_until_monday = (7 - d.weekday()) % 7
    first_monday = d + timedelta(days=days_until_monday)
    return first_monday + timedelta(weeks=2)

# Build the fetch schedule — one week per month, 2023
YEAR = 2023
fetch_schedule = []

for month in range(1, 13):
    start = third_monday(YEAR, month)
    for offset in range(7):
        fetch_schedule.append(start + timedelta(days=offset))

print(f"Fetch schedule: {len(fetch_schedule)} days across 12 months")
print(f"\nSample dates:")
for d in fetch_schedule[:14]:
    print(f"  {d}  (month {d.month})")


In [ ]:
import time

def fetch_all(schedule: list, bands_m: list, delay: float = 2.0) -> None:
    """
    Fetch and cache all days and bands in the schedule.
    Skips already-cached files automatically.
    delay: seconds between requests to be polite to wspr.live
    """
    total = len(schedule) * len(bands_m)
    completed = 0
    skipped = 0
    failed = 0

    for d in schedule:
        date_str = d.strftime("%Y-%m-%d")
        for band_m in bands_m:
            path = day_parquet_path(date_str, band_m)
            if path.exists():
                skipped += 1
                completed += 1
                continue
            
            try:
                fetch_and_cache_day(date_str, band_m)
                time.sleep(delay)
            except Exception as e:
                print(f"  FAILED {date_str} {band_m}m: {e}")
                failed += 1
            
            completed += 1
            remaining = total - completed
            pct = 100 * completed / total
            print(f"  [{pct:.0f}%] {completed}/{total} done, {skipped} skipped, {failed} failed, {remaining} remaining")

# Dry run first — show what would be fetched vs skipped
print("Dry run — checking cache status:")
already_cached = 0
to_fetch = 0
for d in fetch_schedule:
    date_str = d.strftime("%Y-%m-%d")
    for band_m in BANDS:
        path = day_parquet_path(date_str, band_m)
        if path.exists():
            already_cached += 1
        else:
            to_fetch += 1

print(f"  Already cached: {already_cached}")
print(f"  To fetch:       {to_fetch}")
print(f"  Estimated time: {to_fetch * 45 / 60:.0f} minutes")

In [ ]:
# This will run for ~2-3 hours. Safe to interrupt with Ctrl+C and resume later.
# Already-cached files will be skipped automatically on resume.
fetch_all(fetch_schedule, BANDS, delay=2.0)
print("\nFetch complete.")

In [ ]:
# Audit cached WSPR files
missing = []
present = []

for d in fetch_schedule:
    date_str = d.strftime("%Y-%m-%d")
    for band_m in BANDS:
        path = day_parquet_path(date_str, band_m)
        if path.exists():
            present.append((date_str, band_m))
        else:
            missing.append((date_str, band_m))

print(f"Present: {len(present)}/252")
print(f"Missing: {len(missing)}")
if missing:
    print("\nMissing files:")
    for date_str, band_m in missing:
        print(f"  {date_str} {band_m}m")

In [ ]:
# Re-fetch the one missing file
print("Re-fetching missing files...")
for date_str, band_m in missing:
    print(f"  {date_str} {band_m}m")
    fetch_and_cache_day(date_str, band_m, force=True)

print("Done.")